In [ ]:
# %pip install requests numpy Pillow

Note: you may need to restart the kernel to use updated packages.


In [10]:
import os
from getpass import getpass
from pathlib import Path
import requests
import numpy as np

BASE_URL = "https://ai.healtricsapp.com"
API_KEY = os.getenv("ONENOTEAI_API_KEY") or getpass("Enter OneNoteAI API key manually (hidden): ")
if not API_KEY:
    raise ValueError("An API key is required.")

SESSION = requests.Session()
SESSION.headers.update({"X-API-Key": API_KEY}) #Attach the API key to every request because SESSION is used to send requests (eg post) so no need to readd that eash time you are sendinf a request
TIMEOUT = (10, 300)  # connection, read (seconds) #10 second: Maximum time allowed to establish the connection, 300 seconds: Maximum period of inactivity while waiting for response data (Models take time)
print("Gateway configured:", BASE_URL)

Gateway configured: https://ai.healtricsapp.com


In [7]:
# # First Test Example
# response = SESSION.post(
#     f"{BASE_URL}/embed",
#     json={"text": "Testing OneNoteAI from Jupyter"},
#     timeout=TIMEOUT,
# )

# response.raise_for_status()

# result = response.json()

# print("Model:", result["model"])
# print("Dimensions:", result["dimensions"])

An Example of what is sent as request
URL: https://ai.healtricsapp.com/embed
Method: POST
Body: {"text": "Hello"}
Header: X-API-Key: <your-secret>

In [12]:
#_post_json is used by  embed() and generate()
def _post_json(path: str, payload: dict) -> dict:
    try:
        response = SESSION.post(
            f"{BASE_URL}{path}", json=payload, timeout=TIMEOUT
        )
        response.raise_for_status()
        return response.json()
    
    except requests.HTTPError as exc:
        status = exc.response.status_code
        detail = exc.response.text[:300]
        raise RuntimeError(f"{path}: HTTP {status}: {detail}") from None
    except requests.RequestException as exc:
        raise RuntimeError(f"{path}: network request failed: {exc}") from exc

Reusable embed, generate and ocr functions using https://ai.healtricsapp.com endpoints /embed, /generate, /ocr

In [ ]:
def embed(text: str) -> np.ndarray:
    #Get a 1D float embedding vector for one text string.
    result = _post_json("/embed", {"text": text})

    #Convert the response to a NumPy array
    vector = np.asarray(result["embedding"], dtype=np.float32)
    
    #The embedding must be one-dimensional (1D) and The embedding must not be empty
    if vector.ndim != 1 or vector.size == 0:
        raise ValueError("Unexpected embedding response shape")
    return vector

In [18]:
v1 = embed("The patient has high blood pressure.")
print(v1.shape)

(1024,)


In [15]:
def generate(prompt: str) -> str:
    #Generate text using the OneNoteAI Qwen vision/language model.
    result = _post_json("/generate", {"prompt": prompt})
    return result["answer"]

In [17]:
generate("Explain what is RAG used for in AI")

'**RAG** stands for **Retrieval-Augmented Generation**. It is a technique used in AI, particularly in large language models (LLMs), to improve the accuracy and relevance of generated responses by combining two key components:\n\n1. **Information Retrieval (IR)** — fetching relevant documents or data from a large knowledge base or database.\n2. **Generative AI (LLM)** — using the retrieved information to generate a natural, contextually appropriate response.\n\n---\n\n### 🎯 What RAG is Used For:\n\nRAG is primarily used to **enhance the performance of LLMs** by:\n\n- **Reducing hallucinations** — LLMs can sometimes invent or fabricate information. RAG ensures the model only generates content based on actual, retrieved data.\n- **Improving factual accuracy** — By grounding responses in real-world or domain-specific knowledge, RAG helps produce more reliable and accurate answers.\n- **Enabling domain-specific knowledge** — RAG allows models to be trained or fine-tuned on specific topics (

In [19]:
generate("EExplain Azure Managed Identity in two sentences.")

'Azure Managed Identity is a service that automatically authenticates your Azure resources (like VMs or apps) to access Azure resources and other services without needing to manage passwords or credentials. It enables seamless, secure access to resources using Azure AD, reducing the risk of credential exposure and simplifying identity management.'

In [20]:
def ocr(image_path: str | Path) -> str:
    #Upload PNG/JPEG/WEBP and return extracted text.

    #Convert the input into a Path object so you can use properties like .name and .suffix (.png)
    image_path = Path(image_path)

    #check for image extension (.png)/mime (image/png)
    mime = {".png": "image/png", ".jpg": "image/jpeg", ".jpeg": "image/jpeg", ".webp": "image/webp"}.get(image_path.suffix.lower())
    if not mime:
        raise ValueError("Use a PNG, JPEG, or WEBP image")
    
    #image_path.stat().st_size returns the file size in bytes : not >5mb
    if image_path.stat().st_size > 5 * 1024 * 1024:
        raise ValueError("Image exceeds the API's 5 MiB limit")
    
    #Open the image in binary mode ("rb" means read binary) and send OCR request. NB: OCR sends an image file, not just a JSON object.
    with image_path.open("rb") as handle:
        response = SESSION.post(
            f"{BASE_URL}/ocr",
            files={"file": (image_path.name, handle, mime)},
            timeout=TIMEOUT,
        )
    if not response.ok:
        raise RuntimeError(f"/ocr: HTTP {response.status_code}: {response.text[:300]}")
    #parses the JSON response and returns only the extracted text
    return response.json()["extracted_text"]